# T2S(Text-to-SQL) 메트릭 데모 — SDK & API

자연어를 SQL로 바꾸는 에이전트용 메트릭을 SDK와 API 두 방식으로 계산한다.

대상 메트릭: `soft_f1`, `component_match`, `ast_valid`, `t2s_faithfulness`, `t2s_consistency`

> 중요: 결과집합은 평가 시점에 SQL을 실행해서 얻는 것이 아니라, 이미 계산된 결과를 데이터로 받는다(예측 결과는 `execution_result`, 정답 결과는 `gold_execution_result`).

## 사전 준비

```bash
uv sync --extra server --extra t2s
```

> `component_match` / `ast_valid`는 SQL 파싱을 위해 `[t2s]` extra(`sqlglot`)가 필요하다.

## 0. 데이터셋

각 항목은 질문, 자연어 응답(`output`), 생성 SQL(`sql`)과 정답 SQL(`gold_sql`), 그리고 미리 계산된 예측/정답 결과집합을 갖는다. 세 항목은 정답·부분정답·오류로 구성해 메트릭 차이를 보인다.

In [ ]:
DATASET = [
    {  # ① 완전 정답
        "input": "급여가 100000을 넘는 직원의 이름을 알려줘",
        "output": "급여가 100000을 넘는 직원은 Alice와 Carol입니다.",
        "sql": "SELECT name FROM employee WHERE salary > 100000",
        "gold_sql": "SELECT name FROM employee WHERE salary > 100000",
        "execution_result": [{"name": "Alice"}, {"name": "Carol"}],
        "gold_execution_result": [{"name": "Alice"}, {"name": "Carol"}],
    },
    {  # ② 부분 정답: 필터 누락으로 행이 더 나오고, 투영 별칭도 다름
        "input": "부서별 활성 직원 수를 알려줘",
        "output": "Eng 3명, HR 2명입니다.",
        "sql": "SELECT dept, COUNT(*) AS c FROM emp GROUP BY dept",
        "gold_sql": "SELECT dept, COUNT(*) FROM emp WHERE active = 1 GROUP BY dept",
        "execution_result": [{"dept": "Eng", "cnt": 3}, {"dept": "HR", "cnt": 2}],
        "gold_execution_result": [{"dept": "Eng", "cnt": 3}],
    },
    {  # ③ 오류: 문법이 깨진 SQL, 빈 결과
        "input": "가장 비싼 제품을 알려줘",
        "output": "결과를 찾지 못했습니다.",
        "sql": "SELECT FROM WHERE",
        "gold_sql": "SELECT name FROM product ORDER BY price DESC LIMIT 1",
        "execution_result": [],
        "gold_execution_result": [{"name": "Laptop"}],
    },
]
for i, row in enumerate(DATASET):
    print(i, "|", row["sql"], "| exec:", row["execution_result"])

---
# Part 1. SDK

In [ ]:
from agent_eval.core.contracts import EvalContext, MetaKey
from agent_eval.judges.backend import FunctionJudge, lexical_overlap_judge
from agent_eval.metrics.t2s import AstValid, ComponentMatch, SoftF1, T2SConsistency, T2SFaithfulness

# 데이터셋 전체를 러너로 집계해 대표값과 95% 신뢰구간(CI)을 구하는 헬퍼.
# API 응답의 "aggregate"와 정확히 같은 계산이다(러너 하나가 두 곳에서 재사용된다).
from agent_eval.core.gate import GatePolicy
from agent_eval.core.suite import Suite
from agent_eval.offline.runner import evaluate


def aggregate(metric, ctxs):
    suite = Suite("demo", metric.name, [metric], GatePolicy())
    return evaluate(suite, ctxs).aggregates[0]


def run_sdk(metric, ctxs):
    """각 항목을 개별 채점한 뒤 데이터셋 집계를 출력한다."""
    print(f"[SDK] {metric.name}")
    for i, ctx in enumerate(ctxs):
        r = metric.score(ctx)
        print(f"  #{i}: score={r.score:.3f}  passed={r.passed}  error={r.error}")
    agg = aggregate(metric, ctxs)
    print(f"  ▶ 집계 value={agg.value:.3f}  95% CI=[{agg.ci_low:.3f}, {agg.ci_high:.3f}]  n={agg.n}")

contexts = [
    EvalContext(
        input=row["input"],
        output=row["output"],
        metadata={
            MetaKey.SQL: row["sql"],
            MetaKey.GOLD_SQL: row["gold_sql"],
            MetaKey.EXECUTION_RESULT: row["execution_result"],
            MetaKey.GOLD_EXECUTION_RESULT: row["gold_execution_result"],
        },
    )
    for row in DATASET
]
judge = FunctionJudge(lexical_overlap_judge)
print("준비 완료:", len(contexts), "개 컨텍스트")

## 1-1. `soft_f1` — 결과집합 부분 정확도

예측 결과집합(`execution_result`)과 정답 결과집합(`gold_execution_result`)의 (열, 값) 사실 단위 F1. 행 순서·중복은 무시되며 누락된 열은 재현율을 깍아먹는다.

In [ ]:
run_sdk(SoftF1(), contexts)

## 1-2. `component_match` — AST 구성요소 일치

예측/정답 SQL을 파싱해 테이블과 투영(projection) 집합의 자카드 유사도를 구한다(진단용). 별칭이 다르면 점수가 낮아진다.

In [ ]:
run_sdk(ComponentMatch(), contexts)

## 1-3. `ast_valid` — SQL 파싱 가능 여부

생성 SQL이 아예 파싱되는지만 보는 값싼 이진 게이트. 세 번째 항목(`SELECT FROM WHERE`)은 실패한다.

In [ ]:
run_sdk(AstValid(), contexts)

## 1-4. `t2s_faithfulness` — 결과 보고 충실도

최종 자연어 응답(`output`)이 예측 결과집합을 충실히 보고하는지 심판이 판단한다. 심판은 원본 행이 아니라 결과집합의 **통계 요약(digest)**을 읽는다.

In [ ]:
run_sdk(T2SFaithfulness(judge), contexts)

## 1-5. `t2s_consistency` — 결과 일관성

응답이 예측 결과집합과 모순되지 않는지 심판이 판단한다.

In [ ]:
run_sdk(T2SConsistency(judge), contexts)

---
# Part 2. API

In [ ]:
# API 파트: 백그라운드 스레드에서 실제 FastAPI 서버를 띄우고, httpx로 진짜 HTTP 요청을 보낸다.
# (환경변수를 설정하지 않으면 서버도 SDK와 동일한 오프라인 스텁 심판을 쓴다 → 점수가 일치한다.)
import threading
import time

import httpx
import uvicorn

from agent_eval.server.app import create_app

PORT = 8079
BASE_URL = f"http://127.0.0.1:{PORT}"

if "server" not in globals():
    server = uvicorn.Server(uvicorn.Config(create_app(), host="127.0.0.1", port=PORT, log_level="warning"))
    threading.Thread(target=server.run, daemon=True).start()
    while not server.started:
        time.sleep(0.1)
print("API 서버 준비 완료:", BASE_URL)
print("health:", httpx.get(f"{BASE_URL}/health").json())

def call_api(path, contexts, params=None):
    """엔드포인트에 contexts를 POST하고, 개별 결과와 집계를 출력한다."""
    payload = {"contexts": contexts}
    if params:
        payload["params"] = params
    resp = httpx.post(BASE_URL + path, json=payload)
    print(f"[API] POST {path} → {resp.status_code}")
    data = resp.json()
    if resp.status_code != 200:
        print("  오류:", data.get("detail"))
        return data
    for i, item in enumerate(data["results"]):
        print(f"  #{i}: score={item['score']:.3f}  passed={item['passed']}  error={item['error']}")
    agg = data["aggregate"]
    print(f"  ▶ 집계 value={agg['value']:.3f}  95% CI=[{agg['ci_low']:.3f}, {agg['ci_high']:.3f}]  n={agg['n']}")
    return data

## 2-1. `POST /t2s/soft_f1`

`metadata`에 예측/정답 결과집합을 담는다.

In [ ]:
ctx = [{"metadata": {"execution_result": r["execution_result"], "gold_execution_result": r["gold_execution_result"]}} for r in DATASET]
call_api("/t2s/soft_f1", ctx)

## 2-2. `POST /t2s/component_match`

`metadata`에 생성/정답 SQL을 담는다.

In [ ]:
ctx = [{"metadata": {"sql": r["sql"], "gold_sql": r["gold_sql"]}} for r in DATASET]
call_api("/t2s/component_match", ctx)

## 2-3. `POST /t2s/ast_valid`

In [ ]:
ctx = [{"metadata": {"sql": r["sql"]}} for r in DATASET]
call_api("/t2s/ast_valid", ctx)

## 2-4. `POST /t2s/faithfulness`

URL은 이미 `t2s` 접두사를 가지므로 경로는 `/t2s/faithfulness`(레지스트리 타입은 `t2s_faithfulness`)다. `output`과 예측 결과집합을 담는다.

In [ ]:
ctx = [{"input": r["input"], "output": r["output"], "metadata": {"execution_result": r["execution_result"]}} for r in DATASET]
call_api("/t2s/faithfulness", ctx)

## 2-5. `POST /t2s/consistency`

In [ ]:
ctx = [{"input": r["input"], "output": r["output"], "metadata": {"execution_result": r["execution_result"]}} for r in DATASET]
call_api("/t2s/consistency", ctx)